# SakThai v7 — Free Colab Training (Qwen2.5-7B QLoRA)

Trains **Nanthasit/sakthai-context-7b-tools-v7** on **Nanthasit/sakthai-combined-v6** (v6.1 cleaned)
using Unsloth QLoRA on Colab's free T4 GPU. Total cost: **$0**.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have your HF **write** token ready (https://huggingface.co/settings/tokens)
3. Runtime → **Run all** — training takes ~60–90 minutes

Every cell is safe to re-run. The model pushes to the Hub at the end, so nothing is lost if Colab disconnects *after* the push cell.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "trl>=0.12.0" peft accelerate bitsandbytes

In [ ]:
# Log in to Hugging Face (paste your WRITE token when prompted)
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")   # or set it in Colab secrets (key icon, left sidebar)
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("HF write token: ")
login(HF_TOKEN)
print("Logged in.")

In [ ]:
# Download the private dataset and render training text with the Qwen chat template.
# Note: OpenAI-format 'arguments' are JSON strings, but Qwen's template serializes
# with | tojson, so we convert them to dicts before rendering.
import json
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

DATASET = "Nanthasit/sakthai-combined-v6"
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

def load_jsonl(fname):
    path = hf_hub_download(DATASET, fname, repo_type="dataset")
    return [json.loads(l) for l in open(path) if l.strip()]

def render(rec):
    msgs = []
    for m in rec["messages"]:
        m = dict(m)
        if m.get("tool_calls"):
            fixed = []
            for tc in m["tool_calls"]:
                fn = dict(tc["function"])
                if isinstance(fn.get("arguments"), str):
                    fn["arguments"] = json.loads(fn["arguments"])
                fixed.append({**tc, "function": fn})
            m["tool_calls"] = fixed
        msgs.append(m)
    return tok.apply_chat_template(msgs, tools=rec.get("tools") or None, tokenize=False)

from datasets import Dataset
train_rows = load_jsonl("data/train.jsonl")
texts = [render(r) for r in train_rows]
train_ds = Dataset.from_dict({"text": texts})
print(f"train examples: {len(train_ds)}")
print(texts[0][:800])

In [ ]:
# Load Qwen2.5-7B-Instruct in 4-bit and attach LoRA
from unsloth import FastLanguageModel

MAX_LEN = 3072
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# Train (~60-90 min on T4)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=MAX_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
stats = trainer.train()
print(stats)

In [ ]:
# Push the LoRA adapter to the Hub  <-- the important cell
ADAPTER_REPO = "Nanthasit/sakthai-context-7b-tools-v7"
model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print(f"Adapter saved: https://huggingface.co/{ADAPTER_REPO}")

In [ ]:
# Quick smoke test: does v7 call tools correctly?
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

tools = [{"type": "function", "function": {"name": "get_weather", "description": "Get current weather",
          "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "City name"}}, "required": ["location"]}}}]
msgs = [
    {"role": "system", "content": "You are SakThai-Agent, Beer's Growth Partner — sharp, calm, and direct. Call tools when needed, answer directly otherwise."},
    {"role": "user", "content": "อากาศที่กรุงเทพเป็นยังไงบ้าง"},
]
inputs = tokenizer.apply_chat_template(msgs, tools=tools, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=120, temperature=0.3)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False))

# Expect a <tool_call>{"name": "get_weather", "arguments": {"location": "Bangkok"}}</tool_call>
# Also try a no-tool prompt like "สวัสดีครับ" — it should answer directly WITHOUT calling a tool.
FastLanguageModel.for_training(model)

## Optional: push merged 16-bit weights

Run this only if you want the standalone `-merged` repo too (needed for GGUF/Ollama conversion later).
Takes ~15 min extra and is memory-tight on free Colab — if it crashes, the adapter above is already safe on the Hub.

In [ ]:
MERGED_REPO = "Nanthasit/sakthai-context-7b-merged-v7"
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"Merged model: https://huggingface.co/{MERGED_REPO}")